# Manual FRED Series Workbench

This notebook is meant for **manual exploration**.

Use it like this:
1. Put a `SERIES_ID` into the input cell.
2. Run the fetch cell.
3. Call the plotting / testing helpers you need.
4. When a series looks useful, call the save helper to write:
   - raw series CSV
   - engineered features CSV
   - markdown metadata note

It also loads your local **FRED-MD** files from:
- `Data/MD-dataset/*.csv`
- `Data/MD-dataset/appendix/*.csv`

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from dotenv import load_dotenv
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_arch

plt.rcParams["figure.figsize"] = (12, 4)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

In [ ]:
# ---- project paths ----
NOTEBOOK_PATH = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_PATH

while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "Data" / "MD-dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "Data" / "MD-dataset").exists():
    raise RuntimeError("Could not find project root. Open the notebook from inside the MLFCS_RLMM_26 repository.")

load_dotenv(PROJECT_ROOT / ".env")
FRED_API_KEY = os.getenv("FRED_API_KEY")
if not FRED_API_KEY:
    raise RuntimeError("FRED_API_KEY missing. Add it to .env in the repository root.")

DATA_ROOT     = PROJECT_ROOT / "Data"
ANALYZED_ROOT = DATA_ROOT / "analyzed"
ANALYZED_ROOT.mkdir(parents=True, exist_ok=True)

MD_ROOT       = PROJECT_ROOT / "Data" / "MD-dataset"
APPENDIX_ROOT = MD_ROOT / "appendix"

print("Project root:  ", PROJECT_ROOT)
print("FRED-MD root:  ", MD_ROOT)
print("Analyzed store:", ANALYZED_ROOT)


In [ ]:
# ---- local FRED-MD loading ----
def read_csv_with_fallback(path: Path, encodings: list[str] | None = None, **kwargs) -> pd.DataFrame:
    if encodings is None:
        encodings = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]

    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc

    raise UnicodeDecodeError(
        last_error.encoding if last_error else "utf-8",
        last_error.object if last_error else b"",
        last_error.start if last_error else 0,
        last_error.end if last_error else 1,
        f"Could not decode {path.name} with any of: {', '.join(encodings)}",
    )
    

def load_fred_md(md_root: Path = MD_ROOT):
    candidates = sorted(md_root.glob("*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No FRED-MD csv found in {md_root}")
    csv_path = candidates[0]
    raw = read_csv_with_fallback(csv_path)

    date_col = raw.columns[0]
    parsed = pd.to_datetime(raw[date_col], errors="coerce")
    first_date_idx = parsed.first_valid_index()
    if first_date_idx is None:
        raise ValueError("Could not find first valid date row in FRED-MD csv")

    tcodes = None
    if first_date_idx > 0:
        meta_rows = raw.iloc[:first_date_idx].copy()
        numeric_row = meta_rows.iloc[0, 1:]
        if numeric_row.notna().all():
            try:
                tcodes = {col: int(float(val)) for col, val in numeric_row.items()}
            except Exception:
                tcodes = None

    data = raw.iloc[first_date_idx:].copy()
    data[date_col] = pd.to_datetime(data[date_col])
    data = data.rename(columns={date_col: "date"}).set_index("date")
    for col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    appendix = None
    appendix_candidates = sorted((md_root / "appendix").glob("*.csv"))
    if appendix_candidates:
        appendix = read_csv_with_fallback(appendix_candidates[0])

    return {
        "csv_path": csv_path,
        "data": data,
        "tcodes": tcodes,
        "appendix": appendix,
    }

md_bundle = load_fred_md()
md_df = md_bundle["data"]
md_tcodes = md_bundle["tcodes"] or {}
md_appendix = md_bundle["appendix"]

print("Loaded FRED-MD:", md_bundle["csv_path"].name)
print("Shape:", md_df.shape)
print("Date range:", md_df.index.min().date(), "->", md_df.index.max().date())
print("Appendix loaded:", md_appendix is not None)

In [ ]:
# ---- FRED API helpers ----
BASE_URL = "https://api.stlouisfed.org/fred"
SESSION = requests.Session()


def fred_get(endpoint: str, **params):
    url = f"{BASE_URL}/{endpoint}"
    payload = {"api_key": FRED_API_KEY, "file_type": "json", **params}
    r = SESSION.get(url, params=payload, timeout=60)
    r.raise_for_status()
    data = r.json()
    if "error_code" in data:
        raise RuntimeError(f"FRED API error {data['error_code']}: {data.get('error_message')}")
    return data


def search_fred(text: str, limit: int = 20, order_by: str = "search_rank") -> pd.DataFrame:
    data = fred_get("series/search", search_text=text, limit=limit, order_by=order_by)
    return pd.DataFrame(data.get("seriess", []))


def get_series_meta(series_id: str) -> dict:
    data = fred_get("series", series_id=series_id)
    items = data.get("seriess", [])
    if not items:
        raise ValueError(f"No metadata found for {series_id}")
    return items[0]


def get_series_release(series_id: str) -> dict | None:
    data = fred_get("series/release", series_id=series_id)
    items = data.get("releases", [])
    return items[0] if items else None


def get_series_observations(series_id: str, observation_start=None, observation_end=None, frequency=None, aggregation_method=None, units=None):
    data = fred_get(
        "series/observations",
        series_id=series_id,
        observation_start=observation_start,
        observation_end=observation_end,
        frequency=frequency,
        aggregation_method=aggregation_method,
        units=units,
    )
    df = pd.DataFrame(data.get("observations", []))
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"])
    for c in ["realtime_start", "realtime_end"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df


def get_series_vintages(series_id: str, limit: int = 10000) -> list[str]:
    data = fred_get("series/vintagedates", series_id=series_id, limit=limit)
    return data.get("vintage_dates", [])

In [ ]:
# ---- transform helpers ----
TCODE_LABELS = {
    1: "level",
    2: "first_difference",
    3: "second_difference",
    4: "log_level",
    5: "log_first_difference",
    6: "log_second_difference",
    7: "change_in_growth_rate",
}


def apply_tcode(series: pd.Series, tcode: int) -> pd.Series:
    s = series.astype(float).copy()
    if tcode == 1:
        return s
    if tcode == 2:
        return s.diff()
    if tcode == 3:
        return s.diff().diff()
    if tcode == 4:
        return np.log(s.where(s > 0))
    if tcode == 5:
        return np.log(s.where(s > 0)).diff()
    if tcode == 6:
        return np.log(s.where(s > 0)).diff().diff()
    if tcode == 7:
        g = s / s.shift(1) - 1.0
        return g.diff()
    raise ValueError(f"Unsupported tcode {tcode}")


def zscore(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    mu = s.mean()
    sigma = s.std()
    if pd.isna(sigma) or sigma == 0:
        return pd.Series(np.nan, index=s.index, name=f"{series.name}_zscore")
    out = (s - mu) / sigma
    out.name = f"{series.name}_zscore" if series.name else "zscore"
    return out


def variance_proxy(series: pd.Series) -> pd.Series:
    # Squared global z-score highlights high-variance episodes over time.
    z = zscore(series)
    vp = z.pow(2)
    vp.name = f"{series.name}_variance_proxy" if series.name else "variance_proxy"
    return vp


def annualized_change(series: pd.Series, periods: int, periods_per_year: int | None = None) -> pd.Series:
    if periods_per_year is None:
        periods_per_year = infer_periods_per_year(series)
    ratio = series / series.shift(periods)
    return ratio.pow(periods_per_year / periods) - 1


def infer_periods_per_year(series: pd.Series) -> int:
    idx = series.dropna().index
    if len(idx) < 3:
        return 12
    median_days = np.median(np.diff(idx.values).astype('timedelta64[D]').astype(int))
    if median_days <= 2:
        return 252
    if median_days <= 10:
        return 52
    if median_days <= 40:
        return 12
    if median_days <= 120:
        return 4
    return 1


def build_feature_frame(series: pd.Series) -> pd.DataFrame:
    ppy = infer_periods_per_year(series)
    out = pd.DataFrame(index=series.index)
    out["level"] = series
    out["diff_1"] = series.diff(1)
    out["diff_12"] = series.diff(12 if ppy >= 12 else min(ppy, len(series)-1))
    out["pct_change_1"] = series.pct_change(1)
    out["pct_change_12"] = series.pct_change(12 if ppy >= 12 else min(ppy, len(series)-1))
    if ppy >= 12:
        out["ann_3m"] = annualized_change(series, 3, ppy)
        out["ann_6m"] = annualized_change(series, 6, ppy)
    elif ppy == 4:
        out["ann_2q"] = annualized_change(series, 2, ppy)
        out["ann_4q"] = annualized_change(series, 4, ppy)
    out["zscore"] = zscore(series)
    out["variance_proxy"] = variance_proxy(series)
    out["dev_from_36_mean"] = series - series.rolling(min(36, max(12, len(series)//4))).mean()
    return out

In [ ]:
# ---- summary / diagnostics ----
def summarize_series(series_id: str, obs_df: pd.DataFrame, meta: dict | None = None, release: dict | None = None, vintages: list[str] | None = None) -> pd.DataFrame:
    s = obs_df.set_index("date")["value"].sort_index() if "date" in obs_df.columns else obs_df["value"].sort_index()
    total = len(s)
    missing = int(s.isna().sum())
    ppy = infer_periods_per_year(s)

    out = {
        "series_id": series_id,
        "title": (meta or {}).get("title"),
        "frequency": (meta or {}).get("frequency"),
        "units": (meta or {}).get("units"),
        "seasonal_adjustment": (meta or {}).get("seasonal_adjustment"),
        "observation_start": (meta or {}).get("observation_start"),
        "observation_end": (meta or {}).get("observation_end"),
        "n_obs": total,
        "n_missing": missing,
        "missing_share": round(missing / total, 4) if total else np.nan,
        "first_valid": s.first_valid_index(),
        "last_valid": s.last_valid_index(),
        "period_covered": f"{s.first_valid_index().date()} -> {s.last_valid_index().date()}" if s.notna().any() else "no_valid_values",
        "periods_per_year_est": ppy,
        "source": (release or {}).get("link") or (release or {}).get("name"),
        "release_name": (release or {}).get("name"),
        "notes": (meta or {}).get("notes"),
        "n_vintage_dates": len(vintages or []),
        "revision_behavior_hint": "has_multiple_vintages" if len(vintages or []) > 1 else "little_or_no_revision_signal",
    }
    return pd.DataFrame([out]).T.rename(columns={0: "value"})


def _clean_for_tests(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()


def adf_result(series: pd.Series):
    s = _clean_for_tests(series)
    if len(s) < 20 or s.nunique() < 2:
        return {"error": "too_few_or_constant"}
    stat, pvalue, usedlag, nobs, crit, _ = adfuller(s, autolag="AIC")
    return {
        "stat": stat, "pvalue": pvalue, "usedlag": usedlag, "nobs": nobs,
        "critical_values": crit,
        "conclusion_5pct": "stationary" if pvalue < 0.05 else "fails_to_reject_unit_root",
    }


def kpss_result(series: pd.Series):
    s = _clean_for_tests(series)
    if len(s) < 20 or s.nunique() < 2:
        return {"error": "too_few_or_constant"}
    try:
        stat, pvalue, lags, crit = kpss(s, regression="c", nlags="auto")
        return {
            "stat": stat, "pvalue": pvalue, "lags": lags, "critical_values": crit,
            "conclusion_5pct": "fails_to_reject_stationarity" if pvalue >= 0.05 else "rejects_stationarity",
        }
    except Exception as e:
        return {"error": str(e)}


def arch_result(series: pd.Series, nlags: int = 12):
    s = _clean_for_tests(series)
    if len(s) < max(30, nlags * 3) or s.nunique() < 2:
        return {"error": "too_few_or_constant"}
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(s, nlags=nlags)
    return {
        "lm_stat": lm_stat, "lm_pvalue": lm_pvalue, "f_stat": f_stat, "f_pvalue": f_pvalue,
        "conclusion_5pct": "heteroskedastic" if lm_pvalue < 0.05 else "no_strong_arch_effect",
    }


def run_basic_tests(series: pd.Series) -> pd.DataFrame:
    rows = {
        "ADF_level": adf_result(series),
        "KPSS_level": kpss_result(series),
        "ARCH_level": arch_result(series),
        "ADF_diff1": adf_result(series.diff()),
        "KPSS_diff1": kpss_result(series.diff()),
        "ARCH_diff1": arch_result(series.diff()),
    }
    return pd.DataFrame(rows).T


# ---- tcode recommendation ----
# recommend_tcode works like this:
#   1. For each of the 7 FRED-MD transformation codes, check whether the transform
#      can be applied (e.g. log needs all-positive values) and whether the result
#      comes out stationary by ADF at 5 %.
#   2. Pick the simplest eligible stationary transform as the primary suggestion.
#   3. Return a one-hot-style table: one row per tcode, columns for can_apply /
#      stationary / suggested so you can see the full picture at a glance.
def recommend_tcode(series: pd.Series) -> pd.DataFrame:
    s = _clean_for_tests(series)
    all_positive = (s > 0).all()

    def _is_stationary(transformed: pd.Series) -> bool:
        r = adf_result(transformed)
        return r.get("conclusion_5pct") == "stationary"

    candidates = {
        1: ("level",            True,          s),
        2: ("first_difference", True,          s.diff()),
        3: ("second_difference",True,          s.diff().diff()),
        4: ("log_level",        all_positive,  np.log(s) if all_positive else s),
        5: ("log_first_diff",   all_positive,  np.log(s).diff() if all_positive else s),
        6: ("log_second_diff",  all_positive,  np.log(s).diff().diff() if all_positive else s),
        7: ("change_in_growth", all_positive,  (s / s.shift(1) - 1).diff() if all_positive else s),
    }

    rows = []
    suggestion_made = False
    for tcode, (label, can_apply, transformed) in candidates.items():
        stationary = _is_stationary(transformed) if can_apply else False
        suggest = (not suggestion_made) and can_apply and stationary
        if suggest:
            suggestion_made = True
        rows.append({
            "tcode": tcode,
            "label": label,
            "can_apply": int(can_apply),
            "stationary_ADF5pct": int(stationary),
            "suggested": int(suggest),
        })

    return pd.DataFrame(rows).set_index("tcode")

In [ ]:
# ---- plotting helpers ----
def plot_series(series: pd.Series, title: str | None = None):
    ax = series.plot(title=title or series.name)
    ax.axhline(series.mean(), linestyle="--", linewidth=1)
    plt.show()


def plot_diff(series: pd.Series, periods: int = 1, title: str | None = None):
    d = series.diff(periods)
    ax = d.plot(title=title or f"{series.name} diff({periods})")
    ax.axhline(0, linestyle="--", linewidth=1)
    plt.show()
    return d


def plot_pct_change(series: pd.Series, periods: int = 1, title: str | None = None):
    c = series.pct_change(periods)
    ax = c.plot(title=title or f"{series.name} pct_change({periods})")
    ax.axhline(0, linestyle="--", linewidth=1)
    plt.show()
    return c


def plot_zscore(series: pd.Series, title: str | None = None):
    z = zscore(series)
    ax = z.plot(title=title or f"{series.name} z-score (full sample)")
    for y in [-2, 0, 2]:
        ax.axhline(y, linestyle="--", linewidth=1)
    plt.show()
    return z


def plot_variance_proxy(series: pd.Series, title: str | None = None):
    vp = variance_proxy(series)
    ax = vp.plot(title=title or f"{series.name} variance proxy (z-score squared)")
    ax.axhline(vp.mean(), linestyle="--", linewidth=1)
    plt.show()
    return vp


def plot_all_basic(series: pd.Series):
    plot_series(series, f"{series.name} level")
    d1 = plot_diff(series, 1, f"{series.name} first difference")
    plot_zscore(d1.dropna(), f"{series.name} first-difference z-score")
    plot_variance_proxy(d1.dropna(), f"{series.name} variance proxy from first difference")

In [ ]:
# ---- FRED-MD comparison helpers ----
# find_similar_in_fred_md scores each FRED-MD series two ways:
# 1. Text score  (weight 0.7): SequenceMatcher ratio on series_id + title vs appendix description.
# 2. Corr score  (weight 0.3): Pearson on first-differenced, z-scored versions of both series.
#    - Both sides are detrended (first diff) and scaled (z-score) before correlating.
#    - This removes spurious 'both trending up' correlations that inflate raw-level similarity.
#    - Correlation is only a tiebreaker; text overlap is the primary signal.

def _appendix_text_columns(df: pd.DataFrame):
    return [c for c in df.columns if df[c].dtype == object]


def _corr_with_md(input_series: pd.Series, md_series_id: str) -> float | None:
    if md_series_id not in md_df.columns:
        return None
    md_raw = md_df[md_series_id].dropna()
    # First-difference and z-score both series to remove trend and scale effects.
    def _prep(s: pd.Series) -> pd.Series:
        d = s.diff().dropna()
        mu, sigma = d.mean(), d.std()
        if sigma == 0 or pd.isna(sigma):
            return d
        return (d - mu) / sigma
    inp = _prep(input_series)
    md_s = _prep(md_raw)
    common = inp.index.intersection(md_s.index)
    if len(common) < 12:
        return None
    try:
        c = float(inp.loc[common].corr(md_s.loc[common]))
        return c if np.isfinite(c) else None
    except Exception:
        return None


def find_similar_in_fred_md(input_series: pd.Series | None = None, series_id: str = "", title: str | None = None, top_n: int = 10) -> pd.DataFrame:
    candidates = []

    # Exact ID match always scores 1.
    if series_id and series_id in md_df.columns:
        corr = _corr_with_md(input_series, series_id) if input_series is not None else None
        candidates.append({
            "md_series": series_id,
            "tcode": md_tcodes.get(series_id),
            "text_score": 1.0,
            "corr": round(corr, 3) if corr is not None else None,
            "combined_score": 1.0,
            "match_type": "exact_id",
            "appendix_preview": "",
        })

    if md_appendix is not None:
        query_text = f"{series_id} {title or ''}".strip().lower()
        obj_cols = _appendix_text_columns(md_appendix)
        for _, row in md_appendix.iterrows():
            parts = [str(row[c]) for c in obj_cols if pd.notna(row[c])]
            blob = " | ".join(parts)
            text_score = SequenceMatcher(None, query_text, blob.lower()).ratio()

            md_series_id = None
            for val in row.values:
                if isinstance(val, str) and val in md_df.columns:
                    md_series_id = val
                    break

            corr = _corr_with_md(input_series, md_series_id) if (input_series is not None and md_series_id) else None
            corr_component = abs(corr) if corr is not None else 0.0
            # Text dominates (0.7); correlation is a tiebreaker (0.3).
            combined = 0.7 * text_score + 0.3 * corr_component

            candidates.append({
                "md_series": md_series_id,
                "tcode": md_tcodes.get(md_series_id) if md_series_id else None,
                "text_score": round(text_score, 3),
                "corr": round(corr, 3) if corr is not None else None,
                "combined_score": round(combined, 3),
                "match_type": "text+corr",
                "appendix_preview": blob[:160],
            })

    if not candidates:
        return pd.DataFrame()

    out = (
        pd.DataFrame(candidates)
        .sort_values("combined_score", ascending=False)
        .drop_duplicates(subset=["md_series"])
        .head(top_n)
        .reset_index(drop=True)
    )
    return out

In [ ]:
# ---- save helpers ----
def slugify(value: str) -> str:
    import re
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-")


def save_series_package(
    series_id: str,
    obs_df: pd.DataFrame,
    meta: dict,
    release: dict | None = None,
    vintages: list[str] | None = None,
    comment: str = "",
    transformation_note: str = "",
    leading_note: str = "",
    useful: str = "maybe",
    tags: list[str] | None = None,
):
    """
    Save a series exploration to Data/analyzed/<series_id>/:
      <slug>_observations.csv   raw observations
      <slug>_features.csv       engineered feature frame
      <slug>_analysis.json      summary + transform recommendation + your notes
    """
    slug    = slugify(series_id)
    out_dir = ANALYZED_ROOT / slug
    out_dir.mkdir(parents=True, exist_ok=True)

    obs    = obs_df.copy().sort_values("date")
    series = obs.set_index("date")["value"]
    series.name = series_id

    features  = build_feature_frame(series)
    summary   = summarize_series(series_id, obs, meta, release, vintages)
    tcode_rec = recommend_tcode(series)
    md_matches = find_similar_in_fred_md(series, series_id=series_id, title=meta.get("title"), top_n=5)

    # --- data files ---
    obs.to_csv(out_dir / f"{slug}_observations.csv", index=False)
    features.to_csv(out_dir / f"{slug}_features.csv")

    # --- build clean analysis JSON ---
    suggested_row   = tcode_rec[tcode_rec["suggested"] == 1]
    suggested_tcode = int(suggested_row.index[0])      if not suggested_row.empty else None
    suggested_label = suggested_row["label"].iloc[0]   if not suggested_row.empty else None

    def _summary_val(key):
        return summary.loc[key, "value"] if key in summary.index else None

    analysis = {
        "series_id":          series_id,
        "title":              meta.get("title", ""),
        "period":             str(_summary_val("period_covered") or ""),
        "frequency":          meta.get("frequency", ""),
        "units":              meta.get("units", ""),
        "seasonal_adjustment": meta.get("seasonal_adjustment", ""),
        "n_obs":              int(_summary_val("n_obs")) if _summary_val("n_obs") is not None else None,
        "missing_share":      float(_summary_val("missing_share")) if _summary_val("missing_share") is not None else None,
        "n_vintage_dates":    len(vintages or []),
        "suggested_transform": {
            "tcode": suggested_tcode,
            "label": suggested_label,
        },
        "user": {
            "useful":               useful,          # "useful" | "maybe" | "useless"
            "comment":              comment,
            "transformation_note":  transformation_note,
            "timing":               leading_note,    # leading / coincident / lagging / structural
            "tags":                 tags or [],
        },
        "release":      (release or {}).get("name", ""),
        "source_link":  (release or {}).get("link", ""),
        "tcode_options": tcode_rec.reset_index().to_dict(orient="records"),
        "fred_md_matches": (
            md_matches[["md_series", "tcode", "combined_score", "corr", "appendix_preview"]]
            .head(5).to_dict(orient="records")
            if not md_matches.empty else []
        ),
    }

    json_path = out_dir / f"{slug}_analysis.json"
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(analysis, f, indent=2, default=str)

    print(f"Saved → {out_dir.relative_to(PROJECT_ROOT)}/")
    return out_dir


# ---- cross-series summary table ----
def load_analyzed_table() -> pd.DataFrame:
    """Read all saved analysis JSONs and return a flat summary DataFrame."""
    rows = []
    for json_path in sorted(ANALYZED_ROOT.glob("**/*_analysis.json")):
        with json_path.open(encoding="utf-8") as f:
            d = json.load(f)
        rows.append({
            "series_id":      d.get("series_id"),
            "title":          d.get("title"),
            "useful":         d.get("user", {}).get("useful"),
            "period":         d.get("period"),
            "frequency":      d.get("frequency"),
            "units":          d.get("units"),
            "suggested_tcode": d.get("suggested_transform", {}).get("tcode"),
            "suggested_label": d.get("suggested_transform", {}).get("label"),
            "comment":        d.get("user", {}).get("comment"),
            "timing":         d.get("user", {}).get("timing"),
            "tags":           ", ".join(d.get("user", {}).get("tags") or []),
             "top_md_match":   ((d.get("fred_md_matches") or [{}])[0]).get("md_series"),
        })
    return pd.DataFrame(rows) if rows else pd.DataFrame()


def show_include_table():
    """Print two tables: series you'd include vs those you'd exclude or are unsure about."""
    df = load_analyzed_table()
    if df.empty:
        print("No analyzed series found in", ANALYZED_ROOT)
        return
    cols     = ["series_id", "title", "suggested_label", "top_md_match", "period", "comment", "tags"]
    include  = df[df["useful"] == "useful"].reset_index(drop=True)
    exclude  = df[df["useful"] != "useful"].reset_index(drop=True)
    display(Markdown("### ✅ Include"))
    display(include[[c for c in cols if c in include.columns]] if not include.empty else Markdown("_none yet_"))
    display(Markdown("### ❌ Exclude / Unsure"))
    display(exclude[["series_id", "title", "useful", "comment"]] if not exclude.empty else Markdown("_none yet_"))


In [ ]:
# =============================================================

# --- [OPTIONAL] don't know the ticker?
# search_results = search_fred("your search words here", limit=10)
# display(search_results[[c for c in ["id","title","frequency","units","observation_start","observation_end"] if c in search_results.columns]])

SERIES_ID = "GFDEGDQ188S"  

# [OPTIONAL] narrow the date window or force a frequency/unit conversion.
# Leave all as None to get the series exactly as FRED publishes it.
# - OBSERVATION_START / END: clip to a date range, e.g. "1990-01-01"
# - TARGET_FREQUENCY:  resample to a different freq,  e.g. "m" monthly, "q" quarterly
# - UNITS:             FRED-side transform,            e.g. "pch" period-% change, "pc1" year-% change
# - AGG_METHOD:        how to collapse when resampling, e.g. "avg", "sum", "eop" (end of period)
OBSERVATION_START = None
OBSERVATION_END   = None
TARGET_FREQUENCY  = None
UNITS             = None
AGG_METHOD        = None

# ------------------------------------------------------------------
# STEP 1 — fetch from FRED
# ------------------------------------------------------------------
meta     = get_series_meta(SERIES_ID)
release  = get_series_release(SERIES_ID)
obs      = get_series_observations(
    SERIES_ID,
    observation_start=OBSERVATION_START,
    observation_end=OBSERVATION_END,
    frequency=TARGET_FREQUENCY,
    aggregation_method=AGG_METHOD,
    units=UNITS,
)
vintages = get_series_vintages(SERIES_ID)
series   = obs.set_index("date")["value"].sort_index()
series.name = SERIES_ID

# ------------------------------------------------------------------
# STEP 2 — plots (always look at data before numbers)
# level: raw series + long-run mean dashed
# first difference: removes a trend; flat around zero = no drift
# z-score of first diff: standardises the diff so you can spot variance bursts
# variance proxy (z² of first diff): episodes above the dashed line = high-variance regimes
# ------------------------------------------------------------------
plot_all_basic(series)

# ------------------------------------------------------------------
# STEP 3 — metadata summary (after plots so you can cross-check visually)
# period_covered: actual data window with valid observations
# missing_share: fraction of NaNs — > 5 % worth investigating
# n_vintage_dates: >1 means the series gets revised after initial release
# ------------------------------------------------------------------
summary_tbl = summarize_series(SERIES_ID, obs, meta, release, vintages)
display(Markdown(f"### {SERIES_ID}: {meta.get('title', '')}"))
display(summary_tbl.loc[[
    "period_covered", "n_obs", "n_missing", "missing_share",
    "frequency", "units", "seasonal_adjustment", "n_vintage_dates",
]])

# ------------------------------------------------------------------
# STEP 4 — stationarity & heteroskedasticity tests
# ADF: p < 0.05 → stationary (unit root rejected)
# KPSS: p < 0.05 → NOT stationary (reject stationarity)
# ARCH: p < 0.05 → time-varying variance (heteroskedastic)
# Ideally: ADF says stationary AND KPSS does not reject stationarity.
# If both disagree it may be a near-unit-root or fractional integration.
# Tests run on raw level and on first difference.
# ------------------------------------------------------------------
display(run_basic_tests(series))

# ------------------------------------------------------------------
# STEP 5 — tcode recommendation
# Checks all 7 FRED-MD transformation codes.
# can_apply=1 means the transform is valid for this series (e.g. log needs positive values).
# stationary_ADF5pct=1 means ADF rejects unit root after that transform.
# suggested=1 is the simplest valid stationary transform — a reasonable default tcode.
# ------------------------------------------------------------------
display(recommend_tcode(series))

# ------------------------------------------------------------------
# STEP 6 — FRED-MD similarity
# Scores every FRED-MD series on text overlap (appendix description vs your series title)
# AND Pearson correlation over the overlapping date window.
# combined_score = 0.7 * text_score + 0.3 * |corr|
# Both series are first-differenced and z-scored before correlating to avoid
# spurious "both trending up" matches. Text overlap is the primary signal.
# The tcode column tells you what transformation FRED-MD applies to that series.
# ------------------------------------------------------------------
display(find_similar_in_fred_md(series, series_id=SERIES_ID, title=meta.get("title"), top_n=10))

# ------------------------------------------------------------------
# [OPTIONAL] engineered feature frame — uncomment to inspect transforms
# ------------------------------------------------------------------
# features = build_feature_frame(series)
# display(features.tail(12))

# ------------------------------------------------------------------
# [OPTIONAL] save everything to disk once you've decided it's useful.
# ------------------------------------------------------------------
COMMENT           = "should use, but it's just a division of two other notions"
TRANSFORMATION_NOTE = "tests suggest non stationarity, but honestly I don't think so cause both should grow at the same inlfation rate"
TIMING_NOTE       = ""    # leading / coincident / lagging / structural
USEFUL            = "probably"
TAGS              = []
save_series_package(
    SERIES_ID, obs, meta,
    release=release, vintages=vintages,
    comment=COMMENT, transformation_note=TRANSFORMATION_NOTE,
    leading_note=TIMING_NOTE, useful=USEFUL, tags=TAGS,
)


In [ ]:
# =============================================================

# --- [OPTIONAL] don't know the ticker?
search_results = search_fred("public debt growth", limit=10)
display(search_results[[c for c in ["id","title","frequency","units","observation_start","observation_end"] if c in search_results.columns]])

SERIES_ID = ""  

# [OPTIONAL] narrow the date window or force a frequency/unit conversion.
# Leave all as None to get the series exactly as FRED publishes it.
# - OBSERVATION_START / END: clip to a date range, e.g. "1990-01-01"
# - TARGET_FREQUENCY:  resample to a different freq,  e.g. "m" monthly, "q" quarterly
# - UNITS:             FRED-side transform,            e.g. "pch" period-% change, "pc1" year-% change
# - AGG_METHOD:        how to collapse when resampling, e.g. "avg", "sum", "eop" (end of period)
OBSERVATION_START = None
OBSERVATION_END   = None
TARGET_FREQUENCY  = None
UNITS             = None
AGG_METHOD        = None

# ------------------------------------------------------------------
# STEP 1 — fetch from FRED
# ------------------------------------------------------------------
# meta     = get_series_meta(SERIES_ID)
# release  = get_series_release(SERIES_ID)
# obs      = get_series_observations(
#     SERIES_ID,
#     observation_start=OBSERVATION_START,
#     observation_end=OBSERVATION_END,
#     frequency=TARGET_FREQUENCY,
#     aggregation_method=AGG_METHOD,
#     units=UNITS,
# )
# vintages = get_series_vintages(SERIES_ID)
# series   = obs.set_index("date")["value"].sort_index()
# series.name = SERIES_ID

# ------------------------------------------------------------------
# STEP 2 — plots (always look at data before numbers)
# level: raw series + long-run mean dashed
# first difference: removes a trend; flat around zero = no drift
# z-score of first diff: standardises the diff so you can spot variance bursts
# variance proxy (z² of first diff): episodes above the dashed line = high-variance regimes
# ------------------------------------------------------------------
# plot_all_basic(series)

# ------------------------------------------------------------------
# STEP 3 — metadata summary (after plots so you can cross-check visually)
# period_covered: actual data window with valid observations
# missing_share: fraction of NaNs — > 5 % worth investigating
# n_vintage_dates: >1 means the series gets revised after initial release
# ------------------------------------------------------------------
# summary_tbl = summarize_series(SERIES_ID, obs, meta, release, vintages)
# display(Markdown(f"### {SERIES_ID}: {meta.get('title', '')}"))
# display(summary_tbl.loc[[
#     "period_covered", "n_obs", "n_missing", "missing_share",
#     "frequency", "units", "seasonal_adjustment", "n_vintage_dates",
# ]])

# ------------------------------------------------------------------
# STEP 4 — stationarity & heteroskedasticity tests
# ADF: p < 0.05 → stationary (unit root rejected)
# KPSS: p < 0.05 → NOT stationary (reject stationarity)
# ARCH: p < 0.05 → time-varying variance (heteroskedastic)
# Ideally: ADF says stationary AND KPSS does not reject stationarity.
# If both disagree it may be a near-unit-root or fractional integration.
# Tests run on raw level and on first difference.
# ------------------------------------------------------------------
# display(run_basic_tests(series))

# ------------------------------------------------------------------
# STEP 5 — tcode recommendation
# Checks all 7 FRED-MD transformation codes.
# can_apply=1 means the transform is valid for this series (e.g. log needs positive values).
# stationary_ADF5pct=1 means ADF rejects unit root after that transform.
# suggested=1 is the simplest valid stationary transform — a reasonable default tcode.
# ------------------------------------------------------------------
# display(recommend_tcode(series))

# ------------------------------------------------------------------
# STEP 6 — FRED-MD similarity
# Scores every FRED-MD series on text overlap (appendix description vs your series title)
# AND Pearson correlation over the overlapping date window.
# combined_score = 0.7 * text_score + 0.3 * |corr|
# Both series are first-differenced and z-scored before correlating to avoid
# spurious "both trending up" matches. Text overlap is the primary signal.
# The tcode column tells you what transformation FRED-MD applies to that series.
# ------------------------------------------------------------------
# display(find_similar_in_fred_md(series, series_id=SERIES_ID, title=meta.get("title"), top_n=10))

# ------------------------------------------------------------------
# [OPTIONAL] engineered feature frame — uncomment to inspect transforms
# ------------------------------------------------------------------
# features = build_feature_frame(series)
# display(features.tail(12))

# ------------------------------------------------------------------
# [OPTIONAL] save everything to disk once you've decided it's useful.
# ------------------------------------------------------------------
# COMMENT           = ""
# TRANSFORMATION_NOTE = ""
# TIMING_NOTE       = ""    # leading / coincident / lagging / structural
# USEFUL            = ""
# TAGS              = []
# save_series_package(
#     SERIES_ID, obs, meta,
#     release=release, vintages=vintages,
#     comment=COMMENT, transformation_note=TRANSFORMATION_NOTE,
#     leading_note=TIMING_NOTE, useful=USEFUL, tags=TAGS,
# )


In [ ]:
# =============================================================

# --- [OPTIONAL] don't know the ticker?
# search_results = search_fred("your search words here", limit=10)
# display(search_results[[c for c in ["id","title","frequency","units","observation_start","observation_end"] if c in search_results.columns]])

SERIES_ID = ""  

# [OPTIONAL] narrow the date window or force a frequency/unit conversion.
# Leave all as None to get the series exactly as FRED publishes it.
# - OBSERVATION_START / END: clip to a date range, e.g. "1990-01-01"
# - TARGET_FREQUENCY:  resample to a different freq,  e.g. "m" monthly, "q" quarterly
# - UNITS:             FRED-side transform,            e.g. "pch" period-% change, "pc1" year-% change
# - AGG_METHOD:        how to collapse when resampling, e.g. "avg", "sum", "eop" (end of period)
OBSERVATION_START = None
OBSERVATION_END   = None
TARGET_FREQUENCY  = None
UNITS             = None
AGG_METHOD        = None

# ------------------------------------------------------------------
# STEP 1 — fetch from FRED
# ------------------------------------------------------------------
# meta     = get_series_meta(SERIES_ID)
# release  = get_series_release(SERIES_ID)
# obs      = get_series_observations(
#     SERIES_ID,
#     observation_start=OBSERVATION_START,
#     observation_end=OBSERVATION_END,
#     frequency=TARGET_FREQUENCY,
#     aggregation_method=AGG_METHOD,
#     units=UNITS,
# )
# vintages = get_series_vintages(SERIES_ID)
# series   = obs.set_index("date")["value"].sort_index()
# series.name = SERIES_ID

# ------------------------------------------------------------------
# STEP 2 — plots (always look at data before numbers)
# level: raw series + long-run mean dashed
# first difference: removes a trend; flat around zero = no drift
# z-score of first diff: standardises the diff so you can spot variance bursts
# variance proxy (z² of first diff): episodes above the dashed line = high-variance regimes
# ------------------------------------------------------------------
# plot_all_basic(series)

# ------------------------------------------------------------------
# STEP 3 — metadata summary (after plots so you can cross-check visually)
# period_covered: actual data window with valid observations
# missing_share: fraction of NaNs — > 5 % worth investigating
# n_vintage_dates: >1 means the series gets revised after initial release
# ------------------------------------------------------------------
# summary_tbl = summarize_series(SERIES_ID, obs, meta, release, vintages)
# display(Markdown(f"### {SERIES_ID}: {meta.get('title', '')}"))
# display(summary_tbl.loc[[
#     "period_covered", "n_obs", "n_missing", "missing_share",
#     "frequency", "units", "seasonal_adjustment", "n_vintage_dates",
# ]])

# ------------------------------------------------------------------
# STEP 4 — stationarity & heteroskedasticity tests
# ADF: p < 0.05 → stationary (unit root rejected)
# KPSS: p < 0.05 → NOT stationary (reject stationarity)
# ARCH: p < 0.05 → time-varying variance (heteroskedastic)
# Ideally: ADF says stationary AND KPSS does not reject stationarity.
# If both disagree it may be a near-unit-root or fractional integration.
# Tests run on raw level and on first difference.
# ------------------------------------------------------------------
# display(run_basic_tests(series))

# ------------------------------------------------------------------
# STEP 5 — tcode recommendation
# Checks all 7 FRED-MD transformation codes.
# can_apply=1 means the transform is valid for this series (e.g. log needs positive values).
# stationary_ADF5pct=1 means ADF rejects unit root after that transform.
# suggested=1 is the simplest valid stationary transform — a reasonable default tcode.
# ------------------------------------------------------------------
# display(recommend_tcode(series))

# ------------------------------------------------------------------
# STEP 6 — FRED-MD similarity
# Scores every FRED-MD series on text overlap (appendix description vs your series title)
# AND Pearson correlation over the overlapping date window.
# combined_score = 0.7 * text_score + 0.3 * |corr|
# Both series are first-differenced and z-scored before correlating to avoid
# spurious "both trending up" matches. Text overlap is the primary signal.
# The tcode column tells you what transformation FRED-MD applies to that series.
# ------------------------------------------------------------------
# display(find_similar_in_fred_md(series, series_id=SERIES_ID, title=meta.get("title"), top_n=10))

# ------------------------------------------------------------------
# [OPTIONAL] engineered feature frame — uncomment to inspect transforms
# ------------------------------------------------------------------
# features = build_feature_frame(series)
# display(features.tail(12))

# ------------------------------------------------------------------
# [OPTIONAL] save everything to disk once you've decided it's useful.
# ------------------------------------------------------------------
# COMMENT           = ""
# TRANSFORMATION_NOTE = ""
# TIMING_NOTE       = ""    # leading / coincident / lagging / structural
# USEFUL            = ""
# TAGS              = []
# save_series_package(
#     SERIES_ID, obs, meta,
#     release=release, vintages=vintages,
#     comment=COMMENT, transformation_note=TRANSFORMATION_NOTE,
#     leading_note=TIMING_NOTE, useful=USEFUL, tags=TAGS,
# )


In [ ]:
# =============================================================

# --- [OPTIONAL] don't know the ticker?
# search_results = search_fred("your search words here", limit=10)
# display(search_results[[c for c in ["id","title","frequency","units","observation_start","observation_end"] if c in search_results.columns]])

SERIES_ID = ""  

# [OPTIONAL] narrow the date window or force a frequency/unit conversion.
# Leave all as None to get the series exactly as FRED publishes it.
# - OBSERVATION_START / END: clip to a date range, e.g. "1990-01-01"
# - TARGET_FREQUENCY:  resample to a different freq,  e.g. "m" monthly, "q" quarterly
# - UNITS:             FRED-side transform,            e.g. "pch" period-% change, "pc1" year-% change
# - AGG_METHOD:        how to collapse when resampling, e.g. "avg", "sum", "eop" (end of period)
OBSERVATION_START = None
OBSERVATION_END   = None
TARGET_FREQUENCY  = None
UNITS             = None
AGG_METHOD        = None

# ------------------------------------------------------------------
# STEP 1 — fetch from FRED
# ------------------------------------------------------------------
# meta     = get_series_meta(SERIES_ID)
# release  = get_series_release(SERIES_ID)
# obs      = get_series_observations(
#     SERIES_ID,
#     observation_start=OBSERVATION_START,
#     observation_end=OBSERVATION_END,
#     frequency=TARGET_FREQUENCY,
#     aggregation_method=AGG_METHOD,
#     units=UNITS,
# )
# vintages = get_series_vintages(SERIES_ID)
# series   = obs.set_index("date")["value"].sort_index()
# series.name = SERIES_ID

# ------------------------------------------------------------------
# STEP 2 — plots (always look at data before numbers)
# level: raw series + long-run mean dashed
# first difference: removes a trend; flat around zero = no drift
# z-score of first diff: standardises the diff so you can spot variance bursts
# variance proxy (z² of first diff): episodes above the dashed line = high-variance regimes
# ------------------------------------------------------------------
# plot_all_basic(series)

# ------------------------------------------------------------------
# STEP 3 — metadata summary (after plots so you can cross-check visually)
# period_covered: actual data window with valid observations
# missing_share: fraction of NaNs — > 5 % worth investigating
# n_vintage_dates: >1 means the series gets revised after initial release
# ------------------------------------------------------------------
# summary_tbl = summarize_series(SERIES_ID, obs, meta, release, vintages)
# display(Markdown(f"### {SERIES_ID}: {meta.get('title', '')}"))
# display(summary_tbl.loc[[
#     "period_covered", "n_obs", "n_missing", "missing_share",
#     "frequency", "units", "seasonal_adjustment", "n_vintage_dates",
# ]])

# ------------------------------------------------------------------
# STEP 4 — stationarity & heteroskedasticity tests
# ADF: p < 0.05 → stationary (unit root rejected)
# KPSS: p < 0.05 → NOT stationary (reject stationarity)
# ARCH: p < 0.05 → time-varying variance (heteroskedastic)
# Ideally: ADF says stationary AND KPSS does not reject stationarity.
# If both disagree it may be a near-unit-root or fractional integration.
# Tests run on raw level and on first difference.
# ------------------------------------------------------------------
# display(run_basic_tests(series))

# ------------------------------------------------------------------
# STEP 5 — tcode recommendation
# Checks all 7 FRED-MD transformation codes.
# can_apply=1 means the transform is valid for this series (e.g. log needs positive values).
# stationary_ADF5pct=1 means ADF rejects unit root after that transform.
# suggested=1 is the simplest valid stationary transform — a reasonable default tcode.
# ------------------------------------------------------------------
# display(recommend_tcode(series))

# ------------------------------------------------------------------
# STEP 6 — FRED-MD similarity
# Scores every FRED-MD series on text overlap (appendix description vs your series title)
# AND Pearson correlation over the overlapping date window.
# combined_score = 0.7 * text_score + 0.3 * |corr|
# Both series are first-differenced and z-scored before correlating to avoid
# spurious "both trending up" matches. Text overlap is the primary signal.
# The tcode column tells you what transformation FRED-MD applies to that series.
# ------------------------------------------------------------------
# display(find_similar_in_fred_md(series, series_id=SERIES_ID, title=meta.get("title"), top_n=10))

# ------------------------------------------------------------------
# [OPTIONAL] engineered feature frame — uncomment to inspect transforms
# ------------------------------------------------------------------
# features = build_feature_frame(series)
# display(features.tail(12))

# ------------------------------------------------------------------
# [OPTIONAL] save everything to disk once you've decided it's useful.
# ------------------------------------------------------------------
# COMMENT           = ""
# TRANSFORMATION_NOTE = ""
# TIMING_NOTE       = ""    # leading / coincident / lagging / structural
# USEFUL            = ""
# TAGS              = []
# save_series_package(
#     SERIES_ID, obs, meta,
#     release=release, vintages=vintages,
#     comment=COMMENT, transformation_note=TRANSFORMATION_NOTE,
#     leading_note=TIMING_NOTE, useful=USEFUL, tags=TAGS,
# )
